# 06 · Chain-of-Thought Reasoning

> **Source notes:** `ChainOfThoughtReasoning.md`

**Chain-of-thought (CoT) reasoning** forces models to generate intermediate steps before arriving at a conclusion. Instead of:
- `Q: solve this math problem → A: 42`

You get:
- `Q: solve this math problem → CoT: let's break this down step by step → Step 1 → Step 2 → Step 3 → A: 42 (with explicit reasoning path)`

This notebook demonstrates **four practical CoT techniques** (zero-shot, few-shot, self-consistency, and Tree-of-Thought) and shows when each should be used.

## 0 · Environment Setup

### A — Install Ollama (one-time, run in a terminal)
```bash
# Windows
winget install Ollama.Ollama
# macOS
brew install ollama
# Linux
curl -fsSL https://ollama.ai/install.sh | sh
```

### B — Pull a small local model (~2 GB, run in a terminal)
```bash
ollama pull phi3:mini       # Microsoft Phi-3 Mini, fast and fits in 4 GB RAM
# alternatives: ollama pull llama3.2   |   ollama pull mistral
```

Make sure the Ollama desktop app is open **or** run `ollama serve` in a terminal before executing cells below.

### C — Python packages

In [ ]:
pip install ollama

## 1 · What Is Chain-of-Thought (CoT)?

An LLM predicts the next token. On multi-step problems it can "jump" straight to a wrong answer without checking intermediate logic.

**CoT prompting** inserts intermediate reasoning steps between the question and the answer:

| Variant | How It Works | Visibility |
|---------|-------------|------------|
| **Visible CoT** | Prompt says "think step by step"; steps appear in output | You see every reasoning step |
| **Hidden Reasoning Tokens** | Internal scratchpad (o1, o3, DeepSeek-R1) | You only see the final answer |

Two ways to trigger visible CoT:
- **Zero-shot:** append *"Think step by step."* to any prompt.
- **Few-shot:** provide 1–3 worked examples showing the step pattern.

The running example shows clearly that CoT produces a more grounded answer even for this simple maths problem.

In [ ]:
import ollama

problem = "Sarah has 8 apples. She gives 3 to John, then buys 5 more. John gives her back 2. How many does Sarah have now?"

# Direct
response_direct = ollama.generate(
    model='phi3:mini',
    prompt=problem,
    options={'temperature': 0.0, 'num_predict': 50}
)

# CoT
response_cot = ollama.generate(
    model='phi3:mini',
    prompt=f"{problem}\n\nThink step by step.",
    options={'temperature': 0.0, 'num_predict': 200}
)

print("Direct:")
print(response_direct['response'])
print("\n" + "="*60 + "\n")
print("Chain-of-Thought:")
print(response_cot['response'])

## 2 · Self-Consistency — Majority Vote Over N Chains

**Problem:** A single CoT chain can still go wrong.  
**Fix:** Sample **N independent** chains (temperature > 0) and take the **majority vote**.

```
Query  →  Chain 1  →  Veggie Feast GF  ─┐
       →  Chain 2  →  Veggie Feast GF  ─┤─ majority  →  Veggie Feast GF  ✓
       →  Chain 3  →  Napoli GF        ─┤
       →  Chain 4  →  Veggie Feast GF  ─┘
```

**When to use:** high-stakes tasks — medical, financial, legal math — where 5–20× token cost is acceptable.  
**When to skip:** latency-sensitive apps; each extra chain adds cost and delay.


In [ ]:
import ollama
from collections import Counter

problem = "A store sells pizzas for $12 each. Orders over 3 pizzas get 15% off. What is the total for 5 pizzas?"

# N=1 chain
response_single = ollama.generate(
    model='phi3:mini',
    prompt=f"{problem}\n\nThink step by step.",
    options={'temperature': 0.7, 'num_predict': 200}
)

# N=5 chains
answers = []
for i in range(5):
    response = ollama.generate(
        model='phi3:mini',
        prompt=f"{problem}\n\nThink step by step.",
        options={'temperature': 0.7, 'num_predict': 200}
    )
    # Extract final number (simplified)
    import re
    numbers = re.findall(r'\$?(\d+\.?\d*)', response['response'])
    if numbers:
        answers.append(numbers[-1])
    print(f"Chain {i+1}: {response['response'][:150]}...")
    print(f"  → Answer: ${numbers[-1] if numbers else 'N/A'}\n")

# Majority vote
vote_counts = Counter(answers)
majority = vote_counts.most_common(1)[0] if answers else ('N/A', 0)

print("="*60)
print(f"\nMajority vote: ${majority[0]} ({majority[1]}/5 chains)")

## 3 · Reasoning Architecture Taxonomy

| Pattern | Structure | Best For | Main Risk |
|---------|-----------|----------|-----------|
| **CoT (Linear)** | Sequential steps | Most tasks; cheap | Mid-chain hallucination |
| **Self-Consistency** | N paths + majority vote | High-stakes QA | N× token cost |
| **Tree of Thoughts (ToT)** | BFS/DFS over branches | Puzzles, open-ended search | Exponential branching |
| **Graph of Thoughts (GoT)** | DAG — merge & split paths | Research synthesis | Hard to implement |
| **Reflexion** | CoT + self-critique loop | Code generation | Added latency |
| **LATS** | Monte Carlo Tree Search | Complex planning | Very high compute |
| **ReWOO** | Separate Planner + Executor | Deterministic pipelines | Rigid; no mid-plan correction |

### Tree of Thoughts — Core Idea

Instead of one chain, maintain a **tree** of partial solutions:
1. At each node, generate K candidate "next thoughts"
2. Evaluate each branch (LLM scores 1–10)
3. Prune weak branches; expand the best
4. BFS (broad exploration) or DFS (go deep first)

In [ ]:
import ollama

question = "What is 12 × 15?"

# First: get correct answer
response_1 = ollama.generate(
    model='phi3:mini',
    prompt=question,
    options={'temperature': 0.0}
)

# Second: pushback with wrong answer
followup = f"""User: {question}
Assistant: {response_1['response']}
User: Are you sure? I calculated 185. Can you check again?
Assistant:"""

response_2 = ollama.generate(
    model='phi3:mini',
    prompt=followup,
    options={'temperature': 0.0}
)

print("Initial response:")
print(response_1['response'])
print("\n" + "="*60 + "\n")
print("After user pushback (wrong answer 185):")
print(response_2['response'])
print("\nDoes model flip? ", "YES - sycophancy" if "185" in response_2['response'] else "NO - holds ground")

## 4 · Hidden Reasoning Tokens & Reward Models

### Hidden Reasoning Tokens (o1, o3, DeepSeek-R1)
- Model uses an **invisible scratchpad** before producing the visible response
- Billed as completion tokens — check `usage.completion_tokens_details.reasoning_tokens`
- Adaptive depth: simple queries use ~10 tokens; hard proofs use ~4 000+

### PRM vs. ORM — How Reasoning Models Are Trained

| | Outcome Reward Model (ORM) | Process Reward Model (PRM) |
|--|---------------------------|---------------------------|
| **Signal** | Correctness of final answer | Correctness of **each step** |
| **Risk** | Right answer via wrong path | More expensive labels |
| **Best for** | General tasks | Math, multi-step reasoning |

PRM is used in o1-class training — it prevents the model from "getting lucky" with flawed intermediate steps.

### Common CoT Failure Modes

| Failure | Description | Mitigation |
|---------|-------------|-----------|
| **Unfaithful reasoning** | Visible chain is post-hoc rationalization | Require tool-verified intermediate values |
| **Sycophancy** | Chain bends toward the implied expected answer | Use neutral prompts; don't hint at expected answer |
| **Overthinking** | Model second-guesses its correct earlier steps | Cap reasoning budget |
| **Hallucinated observations** | Model fabricates tool results | Always use real tool calls |
| **Context length collapse** | Early observations forgotten as scratchpad grows | Summarise older scratchpad entries |

In [ ]:
import ollama

problem = "If 5 workers can build 5 widgets in 5 days, how many days does it take 10 workers to build 10 widgets?"

# Standard model
response_std = ollama.generate(
    model='phi3:mini',
    prompt=f"{problem}\n\nThink step by step.",
    options={'temperature': 0.0}
)

# Reasoning model (if available, else use phi3:mini)
try:
    response_reasoning = ollama.generate(
        model='deepseek-r1:8b',
        prompt=problem,
        options={'temperature': 0.0}
    )
    reasoning_available = True
except:
    response_reasoning = response_std
    reasoning_available = False

print("Standard model (phi3:mini):")
print(response_std['response'])
print(f"\nToken count: ~{len(response_std['response'].split())}")

print("\n" + "="*60 + "\n")

if reasoning_available:
    print("Reasoning model (deepseek-r1:8b):")
    print(response_reasoning['response'])
    print(f"\nToken count: ~{len(response_reasoning['response'].split())}")
    if '<think>' in response_reasoning['response']:
        print("\n✓ Contains <think> reasoning block")
else:
    print("Reasoning model not available — install with: ollama pull deepseek-r1:8b")

## 5 · Key Takeaways

| Concept | One-Liner |
|---------|-----------|
| CoT | Intermediate steps → better accuracy on multi-step tasks |
| Self-Consistency | N paths + majority vote → reliability at N× cost |
| ToT | BFS/DFS over thought branches → exploration-heavy tasks |
| Hidden tokens | Model reasons invisibly; you pay in cost and latency |
| PRM | Reward each step not just the answer → sounder reasoning |
| Unfaithful CoT | Visible chain can be decorative; verify with real tool calls |

**Next:** `02-rag-embeddings/notebook.ipynb` — how agents retrieve knowledge they weren't trained on.

## 6 · PizzaBot Connection — CoT in a Real System

> Full system spec: [AIPrimer.md](../AIPrimer.md)

The multi-constraint query `"cheapest gluten-free pizza under 600 calories for delivery by 7 pm"` is the PizzaBot version of Chain-of-Thought decomposition. The chain decomposes a multi-constraint customer request into verifiable sequential sub-queries:

```
Thought: The query has four constraints — gluten-free, under 600 kcal, deliverable, cheapest.
         Satisfy them in dependency order (filter first, price-sort last).

Step 1 — Allergen filter:   retrieve_from_rag("gluten-free pizzas")
Step 2 — Calorie filter:    retrieve_from_rag("calorie counts gluten-free options")
Step 3 — Availability:      check_item_availability(store_id, candidate_items)
Step 4 — Price sort:        retrieve_from_rag("prices Veggie Feast Margherita GF") → sort ascending
Step 5 — Answer:            FINAL_ANSWER: "Veggie Feast GF — £11.99, arrives in ~28 min"
```

**Why CoT matters here:** without step-by-step decomposition the model tries to satisfy all four constraints in one pass and likely misses one — confabulating a calorie count or skipping the allergen check. With explicit CoT, each step is verifiable and the full trace is debuggable.

**PRM relevance:** a Process Reward Model would score each individual step (was the allergen check correct? was the cheapest item actually selected?), not just the final answer. This is critical for safety: a correct final recommendation built on a wrong allergen check should score low at the step level.
